In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.ticker import PercentFormatter


In [ ]:
matches = pd.read_excel('Match_Data_for_Case_Study_Final_(2).xlsx', sheet_name = 1)
clients = pd.read_excel('Match_Data_for_Case_Study_Final_(2).xlsx', sheet_name = 2)
providers = pd.read_excel('Match_Data_for_Case_Study_Final_(2).xlsx', sheet_name = 3)

In [ ]:
matches = matches[matches['MATCH_REQUEST_CREATED_AT'] > '2021-06-30'].copy()
matches['accepted_no_decline'] = np.where((~matches['ACCEPTED_AT'].isna()) & (matches['DECLINED_AT'].isna()), 1, 0)


matches['has_session'] = np.where(matches['CLIENT_PROVIDER_FIRST_DATE_OF_SERVICE'].isna(), 0 , 1)

clients_grouped = matches.groupby(['PERSONA_CLIENT_ID', 'MATCH_REQUEST_CREATED_AT'],
                                  as_index = False).agg(had_session = ('has_session', 'max'), accepted = ('accepted_no_decline', 'max'), direct = ('IS_DIRECT_SCHEDULED_BY_CLIENT', 'max') )

clients_grouped = clients_grouped.merge(clients, on = 'PERSONA_CLIENT_ID', how = 'left')


In [ ]:
clients_grouped.groupby(['SESSION_PREFERENCE']).agg(count = ('PERSONA_CLIENT_ID', 'nunique'))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

clients_grouped['MATCH_REQUEST_CREATED_AT'] = pd.to_datetime(clients_grouped['MATCH_REQUEST_CREATED_AT'])

clients_grouped['YearMonth'] = clients_grouped['MATCH_REQUEST_CREATED_AT'].dt.to_period('M')

had_session_rate_per_month = clients_grouped.groupby('YearMonth')['had_session'].mean().reset_index()
sessions_per_month = clients_grouped.groupby('YearMonth')['PERSONA_CLIENT_ID'].nunique().reset_index()

had_session_rate_per_month['YearMonth'] = had_session_rate_per_month['YearMonth'].dt.strftime('%b %Y')
sessions_per_month['YearMonth'] = sessions_per_month['YearMonth'].dt.strftime('%b %Y')
had_session_rate_per_month['had_session'] = had_session_rate_per_month['had_session'] * 100

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot the Number of Sessions on the first y-axis
ax1.plot(sessions_per_month['YearMonth'], sessions_per_month['PERSONA_CLIENT_ID'], label='Number of Sessions', marker='s', color='red')
ax1.set_xlabel('Date')
ax1.set_ylabel('Number of NCRs', color='red')
ax1.set_ylim(0, 12000)  # Set appropriate limits for the session rate
ax1.tick_params(axis='y', labelcolor='red')

plt.xticks(rotation=45)

# Create a second y-axis for the Had Session Rate
ax2 = ax1.twinx()  
ax2.plot(had_session_rate_per_month['YearMonth'], had_session_rate_per_month['had_session'], label='Had Session Rate', marker='o', color='blue')
ax2.set_ylabel('Conversion Rate (%)', color='blue')  
ax2.tick_params(axis='y', labelcolor='blue')
ax2.yaxis.set_major_formatter(PercentFormatter())

ax2.set_ylim(0, 15)  # Set appropriate limits for the session rate

plt.title('Conversion Rate and Number of NCRs')
ax1.grid(True)

fig.tight_layout()  # Adjust layout to make room for the labels
plt.show()

In [ ]:
test = clients_grouped[clients_grouped['had_session'] == 1 ]
session_preference_stats = test.groupby('SESSION_PREFERENCE')['direct'].mean().reset_index()
session_preference_stats = session_preference_stats[session_preference_stats['SESSION_PREFERENCE'].isin(['inPerson', 'video'])]

session_preference_stats['direct'] = session_preference_stats['direct'] * 100

plt.figure(figsize=(8, 6))
plt.bar(session_preference_stats['SESSION_PREFERENCE'], session_preference_stats['direct'], color='blue')

plt.xlabel('Session Preference')
plt.ylabel('Scheduling Rate (%)')
plt.title('Scheduling Rate by Session Preference')
plt.ylim(0, 25)  # Set y-axis limits to percentage range (0 to 100)
plt.gca().yaxis.set_major_formatter(PercentFormatter())

plt.grid(True, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
test = clients_grouped[clients_grouped['accepted'] == 1]
session_preference_stats = test.groupby('SESSION_PREFERENCE')['had_session'].mean().reset_index()
session_preference_stats = session_preference_stats[session_preference_stats['SESSION_PREFERENCE'].isin(['inPerson', 'video'])]

session_preference_stats['had_session'] = session_preference_stats['had_session'] * 100

plt.figure(figsize=(8, 6))
plt.bar(session_preference_stats['SESSION_PREFERENCE'], session_preference_stats['had_session'], color='blue')

plt.xlabel('Session Preference')
plt.ylabel('Conversion Rate (%)')
plt.title('Conversion Rate Given They Are Accepted')
plt.ylim(0, 25)  # Set y-axis limits to percentage range (0 to 100)
plt.gca().yaxis.set_major_formatter(PercentFormatter())

plt.grid(True, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
session_preference_stats = clients_grouped.groupby('SESSION_PREFERENCE')['had_session'].mean().reset_index()
session_preference_stats = session_preference_stats[session_preference_stats['SESSION_PREFERENCE'].isin(['inPerson', 'video'])]

session_preference_stats['had_session'] = session_preference_stats['had_session'] * 100

plt.figure(figsize=(8, 6))
plt.bar(session_preference_stats['SESSION_PREFERENCE'], session_preference_stats['had_session'], color='blue')

plt.xlabel('Session Preference')
plt.ylabel('Conversion Rate (%)')
plt.title('Conversion Rate by Session Preference')
plt.ylim(0, 25)  # Set y-axis limits to percentage range (0 to 100)
plt.gca().yaxis.set_major_formatter(PercentFormatter())

plt.grid(True, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
client_source_stats = clients_grouped.groupby('CLIENT_SOURCE')['had_session'].mean().reset_index()
client_source_stats['had_session'] = client_source_stats['had_session'] * 100

client_source_stats = client_source_stats.sort_values('had_session')

plt.figure(figsize=(8, 6))
plt.bar(client_source_stats['CLIENT_SOURCE'], client_source_stats['had_session'], color='blue')

plt.xlabel('Client Source')
plt.ylabel('Conversion Rate (%)')
plt.title('Conversion Rate by Client Source')
plt.ylim(0, 50)

plt.gca().yaxis.set_major_formatter(PercentFormatter())
plt.grid(True, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
df = matches.merge(providers, on = 'PERSONA_PROVIDER_ID', how = 'left')
gender_stats = df.groupby('EDUCATION_LEVEL')['has_session'].mean().reset_index()
gender_stats = gender_stats[gender_stats['EDUCATION_LEVEL'].isin(['Masters', 'Psychologist'])]


# Step 2: Convert had_session rate to percentage
gender_stats['has_session'] = gender_stats['has_session'] * 100

# Step 3: Plot the data
plt.figure(figsize=(8, 6))
plt.bar(gender_stats['EDUCATION_LEVEL'], gender_stats['has_session'], color='blue')

# Step 4: Add labels and title
plt.xlabel('Education Level')
plt.ylabel('Conversion Rate (%)')
plt.title('Provider Conversion Rate by Education Level')
plt.ylim(0, 8)  # Set y-axis limits to percentage range

# Step 5: Format the y-axis to show percentages
plt.gca().yaxis.set_major_formatter(PercentFormatter())

# Step 6: Add grid for better readability
plt.grid(True, axis='y')

# Step 7: Display the plot
plt.tight_layout()
plt.show()

In [ ]:
df = df.sort_values('MATCH_REQUEST_CREATED_AT')
df['clients_at_time_of_match'] = df.groupby(['PERSONA_PROVIDER_ID'])['has_session'].cumsum()

In [ ]:
# Assuming your DataFrame is named 'df'

# Step 1: Create buckets for provider_capacity
bins = [0, 9, 19, 29,100]  # Define the bucket ranges
labels = ['0-9', '10-19', '20-29', '30+']  # Define labels for the buckets
df['capacity_bucket'] = pd.cut(df['clients_at_time_of_match'], bins=bins, labels=labels, right=False)

# Step 2: Group by the provider_capacity buckets and calculate the mean had_session rate
capacity_stats = df.groupby('capacity_bucket', observed=True)['has_session'].mean().reset_index()

# Step 3: Convert had_session rate to percentage
capacity_stats['has_session'] = capacity_stats['has_session'] * 100

# Step 4: Plot the data
plt.figure(figsize=(8, 6))
plt.bar(capacity_stats['capacity_bucket'], capacity_stats['has_session'], color='blue')

# Step 5: Add labels and title
plt.xlabel('Number of Clients a Provider Has At Time of Match')
plt.ylabel('Conversion Rate (%)')
plt.title('Conversion Rate by Current Provider Capacity')
plt.ylim(0, 10)  # Set y-axis limits to percentage range

# Step 6: Format the y-axis to show percentages
plt.gca().yaxis.set_major_formatter(PercentFormatter())

# Step 7: Add grid for better readability
plt.grid(True, axis='y')

# Step 8: Display the plot
plt.tight_layout()
plt.show()